# Временный тест эмбеддингов: RuModernBERT-base и BERTA на соусах

Цель: быстро проверить, дают ли `deepvk/RuModernBERT-base` и `sergeyzh/BERTA` полезный сигнал на этапе генерации пар SKU для category-run `sauces`.

Это временный исследовательский notebook: он не меняет основной `01_candidate_generation.ipynb`, не перетирает `candidates_sauces.csv` и пишет только отдельные `tmp_*` артефакты в `research/dedup/data/embedding_recall_runs/`.

## Что считаем

- Время построения embeddings.
- Размерность embeddings и число candidate pairs после FAISS top-k.
- Recall по уже размеченным positive-парам из `labeling_sauces.csv`, если файл есть.
- Overlap и top-примеры уникальных пар между двумя моделями.

Важно: текущий gold-set собран из старого candidate pool, поэтому это не финальная честная оценка модели. Но как smoke-test для выбора следующего эмбеддера — достаточно полезно.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import gc
import os
from pathlib import Path
import re
import sys
import time

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import (
    CandidateGenerationConfig,
    FaissCandidateGenerationConfig,
    generate_faiss_candidate_pairs,
    prepare_product_records,
    resolve_category_run,
    resolve_run_paths,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

## Конфигурация

По умолчанию notebook берёт только первые 1000 records по соусам, чтобы не ждать вечность. Для полного прогона поставьте `DEDUP_TMP_RECORD_LIMIT=` пустым или задайте большее число.

In [ ]:
CATEGORY_RUN = resolve_category_run("sauces")
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
PRODUCTS_TABLE = "mpstats_products"
OUTPUT_DIR = RUN_PATHS.data_dir / "embedding_recall_runs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K = int(os.environ.get("DEDUP_TMP_TOP_K", "20"))
MAX_CANDIDATES_RAW = os.environ.get("DEDUP_TMP_MAX_CANDIDATES", "").strip()
MAX_CANDIDATES = int(MAX_CANDIDATES_RAW) if MAX_CANDIDATES_RAW else None
RECORD_LIMIT_RAW = os.environ.get("DEDUP_TMP_RECORD_LIMIT", "1000").strip()
RECORD_LIMIT = int(RECORD_LIMIT_RAW) if RECORD_LIMIT_RAW else None
BATCH_SIZE = int(os.environ.get("DEDUP_TMP_EMBEDDING_BATCH_SIZE", "24"))
MAX_LENGTH = int(os.environ.get("DEDUP_TMP_MAX_LENGTH", "128"))
DEVICE_OVERRIDE = os.environ.get("DEDUP_TMP_DEVICE", "auto").strip().lower()

MODEL_CONFIGS = [
    {
        "label": "RuModernBERT-base",
        "model_id": "deepvk/RuModernBERT-base",
        "prefix": "",
        "note": "base masked-LM encoder; проверяем как raw encoder + mean pooling",
    },
    {
        "label": "BERTA",
        "model_id": "sergeyzh/BERTA",
        "prefix": os.environ.get("DEDUP_TMP_BERTA_PREFIX", "search_document: "),
        "note": "sentence-embedding model; default prefix можно заменить через DEDUP_TMP_BERTA_PREFIX",
    },
]

print("category_run:", CATEGORY_RUN.slug)
print("top_k:", TOP_K)
print("record_limit:", RECORD_LIMIT)
print("batch_size:", BATCH_SIZE)
print("max_length:", MAX_LENGTH)
print("output_dir:", OUTPUT_DIR)

## Загрузка соусов из DuckDB

In [ ]:
def resolve_duckdb_path(project_root: Path) -> Path:
    env_path = os.environ.get("MPSTATS_DUCKDB_PATH")
    candidates = [
        Path(env_path).expanduser() if env_path else None,
        project_root / "mpstats.duckdb",
        Path.home() / "Desktop" / "mpstats" / "mpstats.duckdb",
    ]
    for candidate in candidates:
        if candidate and candidate.exists():
            return candidate
    raise FileNotFoundError(
        "DuckDB-куб не найден. Задайте MPSTATS_DUCKDB_PATH=/absolute/path/to/mpstats.duckdb"
    )


def quote_duckdb_name(name: str) -> str:
    return '"' + name.replace('"', '""') + '"'


DB_PATH = resolve_duckdb_path(PROJECT_ROOT)
print("db_path:", DB_PATH)

with duckdb.connect(str(DB_PATH), read_only=True) as con:
    columns = set(con.execute(f"DESCRIBE {quote_duckdb_name(PRODUCTS_TABLE)}").fetchdf()["column_name"].astype(str))
    if "__project_name" not in columns:
        raise RuntimeError("Для sauces category-run нужен project-фильтр, но в кубе нет __project_name.")

    available_categories = con.execute(
        f"""
        SELECT DISTINCT CAST({quote_duckdb_name("Категория")} AS VARCHAR) AS category
        FROM {quote_duckdb_name(PRODUCTS_TABLE)}
        WHERE {quote_duckdb_name("__project_name")} = ?
        ORDER BY 1
        """,
        [CATEGORY_RUN.project_name],
    ).fetchdf()["category"].dropna().tolist()

    real_category = next((category for category in CATEGORY_RUN.category_aliases if category in available_categories), None)
    if real_category is None:
        raise ValueError(
            f"Не нашёл категорию {CATEGORY_RUN.category_aliases} в проекте {CATEGORY_RUN.project_name}. "
            f"Доступные категории: {available_categories[:20]}"
        )

    products_df = con.execute(
        f"""
        SELECT *
        FROM {quote_duckdb_name(PRODUCTS_TABLE)}
        WHERE {quote_duckdb_name("__project_name")} = ?
          AND CAST({quote_duckdb_name("Категория")} AS VARCHAR) = ?
        """,
        [CATEGORY_RUN.project_name, real_category],
    ).fetchdf()

print("real_category:", real_category)
print("raw_rows:", len(products_df))

## Product records и embedding-текст

In [ ]:
feature_config = CandidateGenerationConfig()
product_records = prepare_product_records(products_df, feature_config)
if RECORD_LIMIT is not None:
    product_records = product_records.head(RECORD_LIMIT).copy()


def build_embedding_base_text(records: pd.DataFrame) -> list[str]:
    texts: list[str] = []
    for row in records.itertuples(index=False):
        brand = str(getattr(row, "brand", "") or "").strip()
        title = str(getattr(row, "title", "") or "").strip()
        text = f"{brand} {title}".strip() if brand else title
        texts.append(" ".join(text.split()))
    return texts


base_texts = build_embedding_base_text(product_records)
summary = pd.DataFrame([
    {
        "raw_rows": len(products_df),
        "product_records": len(product_records),
        "unique_articles": product_records["sku"].nunique(dropna=True),
        "marketplaces": product_records["marketplace"].nunique(dropna=True),
        "unique_titles": product_records["title_norm"].nunique(dropna=True),
    }
])
display(summary)
display(product_records[["marketplace", "sku", "brand", "title", "unit_amount", "total_amount", "multipack_count"]].head(10))

## Embedding helper

Для обеих моделей используется `transformers.AutoTokenizer` + `transformers.AutoModel` и mean pooling по `attention_mask`. Это специально простой тест encoder-качества, без файнтюна и без reranker-логики.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


def choose_device() -> str:
    if DEVICE_OVERRIDE and DEVICE_OVERRIDE != "auto":
        return DEVICE_OVERRIDE
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode_with_transformers(model_id: str, texts: list[str], *, prefix: str = "") -> tuple[np.ndarray, dict[str, object]]:
    device = choose_device()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)
    model.to(device)
    model.eval()

    encoded_batches: list[np.ndarray] = []
    started = time.perf_counter()
    prefixed_texts = [prefix + text for text in texts]
    with torch.inference_mode():
        for start in range(0, len(prefixed_texts), BATCH_SIZE):
            batch_texts = prefixed_texts[start : start + BATCH_SIZE]
            inputs = tokenizer(
                batch_texts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            inputs = {key: value.to(device) for key, value in inputs.items()}
            outputs = model(**inputs)
            batch_embeddings = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
            batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
            encoded_batches.append(batch_embeddings.detach().cpu().numpy().astype("float32"))

    elapsed = time.perf_counter() - started
    embeddings = np.vstack(encoded_batches) if encoded_batches else np.empty((0, 0), dtype="float32")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    if device == "mps":
        try:
            torch.mps.empty_cache()
        except AttributeError:
            pass
    gc.collect()

    return embeddings, {"device": device, "embedding_seconds": elapsed, "dimension": embeddings.shape[1] if embeddings.ndim == 2 else 0}

## Gold-set helper

Если есть `labeling_sauces.csv`, считаем приблизительный recall: нашла ли модель уже размеченные positive-пары среди top-k соседей. Это не заменяет новый gold-set, но быстро показывает, насколько генератор пар не теряет известные дубли.

In [ ]:
POSITIVE_LABELS = {"exact_duplicate", "same_product_different_pack"}
RECALL_K_VALUES = [1, 5, 10, TOP_K]


def pair_key(left: object, right: object) -> tuple[str, str]:
    return tuple(sorted((str(left), str(right))))


def candidate_pair_keys(frame: pd.DataFrame, *, max_rank: int | None = None) -> set[tuple[str, str]]:
    data = frame
    if max_rank is not None and "candidate_rank" in data.columns:
        data = data[data["candidate_rank"] <= max_rank]
    return {
        pair_key(row.raw_record_id_a, row.raw_record_id_b)
        for row in data[["raw_record_id_a", "raw_record_id_b"]].itertuples(index=False)
    }


def load_positive_gold(labeling_path: Path, record_ids: set[str]) -> pd.DataFrame:
    if not labeling_path.exists():
        return pd.DataFrame(columns=["raw_record_id_a", "raw_record_id_b", "label"])
    labeling = pd.read_csv(labeling_path)
    if "label" not in labeling.columns:
        return pd.DataFrame(columns=["raw_record_id_a", "raw_record_id_b", "label"])
    gold = labeling[labeling["label"].astype(str).isin(POSITIVE_LABELS)].copy()
    gold = gold[
        gold["raw_record_id_a"].astype(str).isin(record_ids)
        & gold["raw_record_id_b"].astype(str).isin(record_ids)
    ].copy()
    return gold


record_ids = set(product_records["raw_record_id"].astype(str))
positive_gold = load_positive_gold(RUN_PATHS.labeling_path, record_ids)
positive_gold_keys = {
    pair_key(row.raw_record_id_a, row.raw_record_id_b)
    for row in positive_gold[["raw_record_id_a", "raw_record_id_b"]].itertuples(index=False)
}
print("labeling_path:", RUN_PATHS.labeling_path)
print("positive_gold_in_scope:", len(positive_gold_keys))

## Запуск двух моделей

In [ ]:
def safe_model_label(label: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", label).strip("_").lower()


candidate_config = FaissCandidateGenerationConfig(
    top_k=TOP_K,
    max_candidates=MAX_CANDIDATES,
    normalize_vectors=True,
    candidate_features=feature_config,
)

results: list[dict[str, object]] = []
candidate_frames: dict[str, pd.DataFrame] = {}

for config in MODEL_CONFIGS:
    label = config["label"]
    model_id = config["model_id"]
    prefix = config.get("prefix", "")
    print(f"\n=== {label} :: {model_id} ===")
    print("prefix:", repr(prefix))
    started = time.perf_counter()
    try:
        embeddings, embedding_meta = encode_with_transformers(model_id, base_texts, prefix=prefix)
        candidates = generate_faiss_candidate_pairs(product_records, embeddings, candidate_config)
        total_seconds = time.perf_counter() - started
        candidate_frames[label] = candidates

        row = {
            "label": label,
            "model_id": model_id,
            "prefix": prefix,
            "status": "ok",
            "error": "",
            "records": len(product_records),
            "embedding_dim": embedding_meta["dimension"],
            "device": embedding_meta["device"],
            "embedding_seconds": round(float(embedding_meta["embedding_seconds"]), 3),
            "total_seconds": round(float(total_seconds), 3),
            "candidate_pairs": len(candidates),
            "cross_marketplace_pairs": int(candidates["is_cross_marketplace_pair"].sum()) if not candidates.empty else 0,
            "mean_similarity": round(float(candidates["embedding_similarity_score"].mean()), 4) if not candidates.empty else np.nan,
        }

        for k in RECALL_K_VALUES:
            found = len(positive_gold_keys & candidate_pair_keys(candidates, max_rank=k))
            row[f"found_positive_at_{k}"] = found
            row[f"recall_at_{k}"] = round(found / len(positive_gold_keys), 4) if positive_gold_keys else np.nan

        output_path = OUTPUT_DIR / f"tmp_candidates_{safe_model_label(label)}_top{TOP_K}.csv"
        candidates.to_csv(output_path, index=False)
        row["candidate_path"] = str(output_path)
        results.append(row)
        print("candidate_pairs:", len(candidates))
        print("saved:", output_path)
    except Exception as exc:
        total_seconds = time.perf_counter() - started
        results.append(
            {
                "label": label,
                "model_id": model_id,
                "prefix": prefix,
                "status": "error",
                "error": repr(exc),
                "records": len(product_records),
                "total_seconds": round(float(total_seconds), 3),
            }
        )
        print("ERROR:", repr(exc))

results_df = pd.DataFrame(results)
summary_path = OUTPUT_DIR / f"tmp_embedding_test_rumodernbert_berta_sauces_top{TOP_K}.csv"
results_df.to_csv(summary_path, index=False)
print("summary saved:", summary_path)
display(results_df)

## Быстрый график

In [ ]:
plot_df = results_df[results_df["status"].eq("ok")].copy()
if plot_df.empty:
    print("Нет успешных прогонов для графика.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_df.plot.bar(x="label", y="total_seconds", ax=axes[0], legend=False, color="#4C78A8")
    axes[0].set_title("Total seconds")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("seconds")

    recall_col = f"recall_at_{TOP_K}"
    if recall_col in plot_df.columns and plot_df[recall_col].notna().any():
        plot_df.plot.bar(x="label", y=recall_col, ax=axes[1], legend=False, color="#59A14F")
        axes[1].set_title(f"Positive recall@{TOP_K}")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("share")
        axes[1].set_ylim(0, 1)
    else:
        axes[1].axis("off")
        axes[1].text(0.05, 0.5, "Нет positive gold в текущем срезе", fontsize=12)

    plt.tight_layout()
    plt.show()

## Overlap и уникальные top-пары

In [ ]:
def pair_set(frame: pd.DataFrame) -> set[tuple[str, str]]:
    return candidate_pair_keys(frame)


overlap_rows: list[dict[str, object]] = []
labels = list(candidate_frames)
for i, left_label in enumerate(labels):
    for right_label in labels[i + 1 :]:
        left_set = pair_set(candidate_frames[left_label])
        right_set = pair_set(candidate_frames[right_label])
        union_size = len(left_set | right_set)
        overlap_rows.append(
            {
                "left": left_label,
                "right": right_label,
                "left_pairs": len(left_set),
                "right_pairs": len(right_set),
                "intersection": len(left_set & right_set),
                "jaccard": round(len(left_set & right_set) / union_size, 4) if union_size else np.nan,
            }
        )

overlap_df = pd.DataFrame(overlap_rows)
display(overlap_df)

for label, frame in candidate_frames.items():
    others = set().union(*(pair_set(other) for other_label, other in candidate_frames.items() if other_label != label))
    unique_keys = pair_set(frame) - others
    unique = frame[
        frame.apply(lambda row: pair_key(row["raw_record_id_a"], row["raw_record_id_b"]) in unique_keys, axis=1)
    ].sort_values("embedding_similarity_score", ascending=False)
    print(f"\nTop unique pairs for {label}: {len(unique_keys):,} unique pairs")
    display(
        unique[
            [
                "marketplace_a",
                "marketplace_b",
                "sku_a",
                "sku_b",
                "brand_a",
                "brand_b",
                "title_a",
                "title_b",
                "embedding_similarity_score",
                "candidate_rank",
                "is_cross_marketplace_pair",
            ]
        ].head(20)
    )

## Как читать результат

- Если `recall_at_20` ниже E5-памяти/старых прогонов, модель как генератор пар слабее для текущего gold-set.
- Если recall похожий, но уникальные top-пары выглядят руками лучше, стоит добавить эти пары в новый общий pool для разметки.
- Если модель сильно медленнее и не даёт новых хороших пар, не тащим её дальше.

Для полного прогона можно запустить так:

```bash
DEDUP_TMP_RECORD_LIMIT= DEDUP_TMP_TOP_K=20 jupyter notebook notebooks/tmp_embedding_test_rumodernbert_berta_sauces.ipynb
```